# Week 11 - Error Handling and Modular Programming

## Learning objectives

- Getting started with error handling
- Getting started with modular programming

## Overview

In this tutorial we are going to take a step back and focus on two important concepts in programming, namely **error handling** and **modular programming**. We are going to introduce far less new code compared to the previous weeks, given that our development is sufficiently mature at this point, rather we are going to focus on making the code more robust using error handling in the first part of the exercise, and re-organizing the code in a modular manner in the second. Both of these concepts put emphasis on extensibility and robustness, as we shall see.

## Error handling
Error handling is perhaps the single most effective method against runtime errors. Proper error handling can not only save an infinite amount of time, which would otherwise be typically spent bashing one's head firmly against the most solid wall in the vicinity, but can also prevent potential catastrophes, especially for sensitive applications.

In essence, error handling is the proper handling of cases that are considered invalid during the runtime of a program. For instance, you may write a simple function to divide two numbers:

In [ ]:
def divide(a, b):
    return a / b

a = [0] * 10
a[0] = 1
a[100000000] = 1

This innocent-looking function may seem fully functional at first glance; however, consider the following case: what if the user inadvertently or out of pure evilness passes $a \in \mathbb{R} \land a \ne 0$ and $b = 0$? You may have noticed that the domain of validity of the division function implicitly is $a, b \in \mathbb{R} \land b \ne 0$; however, without further instructions, the function `divide` knows nothing of this restriction.

This is where error handling comes in. Add error handling to the function `divide` to *catch* the *case* above.

In [ ]:
def divide(a, b):
    # Code here
    return a / b

You may have guessed that error handling becomes more complex as the code it is being added to increases in complexity. In the following, your task is to add error handling to the real dense and sparse matrices below.

In order to accomplish this task, you must first start by identifying the *invalid* cases, and how to *handle* them.

In [ ]:
import csv
import random

class Mat:
    # Constructor
    # Initializes an empty matrix
    def __init__(self):
        # Initialize row and column size
        # Note that the number of rows and columns are defined with preceding double underscores
        # denoting them as private variables of the class
        self.__n_rows = 0
        self.__n_cols = 0

    # Reinit
    # Resize the matrix and initialize it with random values
    # Note that the resizing and initialization of the matrix depends
    # on its internal structure; therefore, it cannot be implemented here in full
    # except for setting the number of rows and columns
    def reinit(self, n, m):
        self.__n_rows = n
        self.__n_cols = m

    # Entry access through the parentheses operator
    # Returns the value at the given row and column i, j
    # Note that this function depends on the inner workings of the
    # particular matrix format; therefore, it cannot be implemented here.
    def __call__(self, i, j):
        pass

    # Number of rows
    def n_rows(self):
        return self.__n_rows

    # Number of columns
    def n_cols(self):
        return self.__n_cols

    # Set entry
    # Sets the value of the given entry
    def set(self, i, j, val):
        pass

    # Check if entry exists
    def has(self, i, j):
        pass

    # Return the non-zero columns of the given row
    def row(self, i):
        pass

    # Matrix-vector multiplication
    def mult(self, vec):
        # Initialize result vector to zero
        res = [0] * self.__n_rows
        # Loop through rows
        for i in range(0, self.__n_rows):
            # Loop through columns
            for j in range(0, self.__n_cols):
                res[i] += self(i, j) * vec[j]

        return res

    # Write to file
    # Writes the matrix, row by row, in CSV format to disk
    def write(self, filename):
        # Create CSV file
        with open(f"{filename}.csv", mode='w') as file:
            # Create CSV writer
            writer = csv.writer(file, delimiter=',')

            # Loop through rows
            for i in range(0, self.__n_rows):
                row = [0] * self.__n_cols
                for j in range(0, self.__n_cols):
                    row[j] = self(i, j)

                writer.writerow(row)

# Dense matrix
# The data is internally represented as a row-major one-dimensional array, i.e.,
# A = [a00, a01, ..., a10, a11, ..., ..., a(n-1)0, a(n-1)1, ..., a(n-1)(n-1)]
#      ... row 0 ...  ... row 1 ...       ............ row n-1 ............
class DenseMat(Mat):
    # Constructor
    # Initializes an empty matrix
    def __init__(self):
        # Call parent's constructor 
        super().__init__()
        # Initialize data
        # Data represents the matrix entries as a row-major one-dimensional array
        # Note the preceding double underscores, denoting the member variable as private
        self.__data = []

    # Reinit
    # Resize the matrix and initialize it with random values
    def reinit(self, n, m):
        super().reinit(n, m)
        # The matrix has n X m entries; therefore, the data array has a size of n X m
        self.__data = [random.random() for _ in range(0, n * m)]

    # Set entry
    # Sets the value of the given entry
    def set(self, i, j, val):
        self.__data[i * super().n_cols() + j] = val
        
    # Entry access through the brackets operator
    # Returns the value at the given row and column
    # The matrix is stored in row-major format; therefore, the ij entry can be accessed using
    # the index [i * n_cols + j] in the data array
    def __call__(self, i, j):
        return self.__data[i * super().n_cols() + j]

# Coordinate format matrix
class COOMat(Mat):
    # Constructor
    # Initializes an empty matrix
    def __init__(self):
        # Call parent's constructor 
        super().__init__()
        # Initialize data
        # The matrix is represented using three vectors
        # The array of data
        self.__a = []
        # The array of row indices
        self.__i = []
        # The array of column indices
        self.__j = []

    # Reinit
    # Resize the matrix and initialize it with random values
    # n and m represent the number of rows and columns, respectively.
    # nnz_per_row is the number of non-zero elements per row
    def reinit(self, n, m, nnz_per_row):
        super().reinit(n, m)
        # The matrix has nnz_per_row entries per row; therefore, the total number of non-zero
        # entries, and the length of the a, i, and j vectors, is n * nnz_per_row
        nnz = nnz_per_row * n
        self.__a = [0] * nnz
        self.__i = [0] * nnz
        self.__j = [0] * nnz
        # Loop through rows
        for i in range(0, n):
            # Generate nnz_per_row unique column indices
            cols = random.sample(range(0, m), nnz_per_row)
            # Loop through the generated column indices
            for j in range(0, nnz_per_row):
                # The index of the current entry
                index = nnz_per_row * i + j
                # Generate random entry
                self.__a[index] = random.random()
                # Row index
                self.__i[index] = i
                # Column index
                self.__j[index] = cols[j]

    # Set entry
    # Sets the value of the given entry
    # If the entry does not exist, it is appended at the end
    def set(self, i, j, val):
        found = False
        # The total number of non-zero entries
        nnz = len(self.__a)
        for k in range(0, nnz):
            # Check if the row index matches
            if (self.__i[k] == i):
                # Check if the column index matches
                if (self.__j[k] == j):
                    # Set value
                    self.__a[k] = val
                    # Entry found, break
                    found = True
                    break
        # If entry not found
        if (not found):
            # Entry not found, append at the end
            self.__i.append(i)
            self.__j.append(j)
            self.__a.append(val)
        
    # Entry access through the parentheses operator
    # Returns the value at the given row and column
    def __call__(self, i, j):
        # The total number of non-zero entries
        nnz = len(self.__a)
        for k in range(0, nnz):
            # Check if the row index matches
            if (self.__i[k] == i):
                # Check if the column index matches
                if (self.__j[k] == j):
                    # Return the value
                    return self.__a[k]

        # If no non-zero entry at the given row and column indices is found,
        # the entry is zero
        return 0.

    # Matrix-vector multiplication
    def mult(self, vec):
        # Initialize result vector to zero
        res = [0] * self.n_rows()
        nnz = len(self.__a)
        # Loop through non-zeros
        for k in range(0, nnz):
            res[self.__i[k]] += self.__a[k] * vec[self.__j[k]]
        return res

# List of dictionaries real sparse matrix
class LoDMat(Mat):
    # Constructor
    # Initializes an empty matrix
    def __init__(self):
        # Call parent's constructor 
        super().__init__()
        # Initialize data
        # List of dictionaries
        self.__rows = []

    # Reinit
    # Resize the matrix and initialize it with random values
    # n and m represent the number of rows and columns, respectively.
    # nnz_per_row is the number of non-zero elements per row
    def reinit(self, n, m, nnz_per_row):
        super().reinit(n, m)
        # Create a dictionary per row
        self.__rows = [dict() for _ in range(0, n)]
        # Loop through rows
        for i in range(0, n):
            # Generate nnz_per_row unique column indices
            cols = random.sample(range(0, m), nnz_per_row)
            # Loop through the generated column indices
            for j in range(0, nnz_per_row):
                # Generate random entry and add it to the corresponding row
                self.__rows[i][cols[j]] = random.random()

    # Set entry
    # Sets the value of the given entry
    # If the entry does not exist, it is appended at the end
    def set(self, i, j, val):
        self.__rows[i][j] = val
        
    # Entry access through the parentheses operator
    # Returns the value at the given row and column
    def __call__(self, i, j):
        # Entry exists
        if j in self.__rows[i].keys():
            return self.__rows[i][j]
        # Entry does not exist
        else:
            return 0.

    # Check if entry exists
    def has(self, i, j):
        # Entry exists
        if j in self.__rows[i].keys():
            return True
        # Entry does not exist
        else:
            return False

    # Return the non-zero columns of the given row
    def row(self, i):
        return self.__rows[i].keys()

    # Matrix-vector multiplication
    def mult(self, vec):
        # Initialize result vector to zero
        n_rows = self.n_rows()
        res = [0.] * n_rows
        # Loop through rows
        for i in range(0, n_rows):
            # Loop through row non-zero entries
            for j, val in self.__rows[i].items():
                res[i] += val * vec[j]
        return res

Now try and deliberately produce a few examples where undesired operations are performed. Are the errors properly handled?

## Modular programming
The next topic, namely modular programming revolves around the idea that as the program code becomes larger, its management and usage become likewise unwieldy; therefore, it is common practice, and for good reason, to break down and group code that conceptually belongs together into units. Although these units may look rather different from one programming language to the next, the underlying idea remains almost identical. In Python, we speak of **modules**, that encompass code such as functions and classes, and **packages**, that encompass modules.

We have developed, during the previous weeks, a considerable amount of code. You can imagine that the further development of even this moderate number of classes and functions would quickly become cumbersome, were you to undertake the task in a single file. You can therefore hopefully appreciate your next task, which involves the grouping of the code into a sensible module structure. Although such a structure is not unique, meaning that one may come up with multiple ways to organize the same code, each equally valid, one natural way to organize our code is to create a module for complex number and put real and complex matrices into separate **subpackages** as modules, as illustrated below

```
sp
|   __init__.py
│   README.md
│   complex.py
└───mat
|   |   __init__.py
│   │   mat.py
│   │   dense.py
│   │   coo.py
│   │   lod.py
│   
└───cmat
|   |   __init__.py
│   │   cmat.py
│   │   dense.py
│   │   coo.py
│   │   lod.py
```

Note that

- there are three naming layers at play here: the top (root) directory `sp` is the **package**, sub-directories within `sp`, e.g., `mat` are **subpackages**, and individual `.py` files, e.g., `complex.py` and `cmat.py` are **modules**,
- in each directory that should be a (sub)package, a `__init__.py` file must be present. This includes both the top directory and virtually always all of its sub-directories,
- the `__init__.py` files get executed whenever the corresponding (sub)package is imported by a process **for the first time**,
- the filenames indicate the module name and are not necessarily the same as classes and functions that are stored in them, e.g., `sp/mat/dense.py` indicates that there is a module named `dense` under `sp.mat`, which could, however, include various classes and functions with independent names. In our case, we are going to organize the class `DenseMat` in `sp/mat/dense.py`,
- the `README.md` file should provide some basic information about the package and its usage. Here, you can simply write a brief description of the package, e.g., "A Python package for complex numbers and dense and sparse matrices with support for real and complex numbers."

Your task is to create the directory structure above, organize the code into it accordingly. Note that this is the first time we are leaving the safety of Jupyter notebooks. You need to use a text editor in order to create the files.

You can find the rest of the code below.

In [ ]:
import csv
import random

# Complex matrix parent class
class CMat:
    # Constructor
    # Initializes an empty matrix
    def __init__(self):
        # Initialize row and column size
        # Note that the number of rows and columns are defined with preceding double underscores
        # denoting them as private variables of the class
        self.__n_rows = 0
        self.__n_cols = 0

    # Reinit
    # Resize the matrix and initialize it with random values
    # Note that the resizing and initialization of the matrix depends
    # on its internal structure; therefore, it cannot be implemented here in full
    # except for setting the number of rows and columns
    def reinit(self, n, m):
        self.__n_rows = n
        self.__n_cols = m

    # Entry access through the parentheses operator
    # Returns the value at the given row and column i, j
    # Note that this function depends on the inner workings of the
    # particular matrix format; therefore, it cannot be implemented here.
    def __call__(self, i, j):
        pass

    # Number of rows
    def n_rows(self):
        return self.__n_rows

    # Number of columns
    def n_cols(self):
        return self.__n_cols

    # Set entry
    # Sets the value of the given entry
    def set(self, i, j, val):
        pass

    # Matrix-vector multiplication
    def mult(self, vec):
        # Initialize result vector to zero
        res = [Complex(0., 0.)] * self.__n_rows
        # Loop through rows
        for i in range(0, self.__n_rows):
            # Loop through columns
            for j in range(0, self.__n_cols):
                res[i] += self(i, j) * vec[j]

        return res

    # Write to file
    # Writes the matrix, row by row, in CSV format to disk
    def write(self, filename):
        # Create CSV file
        with open(f"{filename}.csv", mode='w') as file:
            # Create CSV writer
            writer = csv.writer(file, delimiter=',')

            # Loop through rows
            for i in range(0, self.__n_rows):
                row = [0] * self.__n_cols
                for j in range(0, self.__n_cols):
                    row[j] = self(i, j)

                writer.writerow(row)

# Dense complex matrix
# The data is internally represented as a row-major one-dimensional array, i.e.,
# A = [a00, a01, ..., a10, a11, ..., ..., a(n-1)0, a(n-1)1, ..., a(n-1)(n-1)]
#      ... row 0 ...  ... row 1 ...       ............ row n-1 ............
class DenseCMat(CMat):
    # Constructor
    # Initializes an empty matrix
    def __init__(self):
        # Call parent's constructor 
        super().__init__()
        # Initialize data
        # Data represents the matrix entries as a row-major one-dimensional array
        # Note the preceding double underscores, denoting the member variable as private
        self.__data = []

    # Reinit
    # Resize the matrix and initialize it with random values
    def reinit(self, n, m):
        super().reinit(n, m)
        # The matrix has n X m entries; therefore, the data array has a size of n X m
        self.__data = [Complex(random.random(), random.random()) for _ in range(0, n * m)]

    # Set entry
    # Sets the value of the given entry
    def set(self, i, j, val):
        self.__data[i * super().n_cols() + j] = val
        
    # Entry access through the brackets operator
    # Returns the value at the given row and column
    # The matrix is stored in row-major format; therefore, the ij entry can be accessed using
    # the index [i * n_cols + j] in the data array
    def __call__(self, i, j):
        return self.__data[i * super().n_cols() + j]

# Coordinate format complex matrix
class COOCMat(CMat):
    # Constructor
    # Initializes an empty matrix
    def __init__(self):
        # Call parent's constructor 
        super().__init__()
        # Initialize data
        # The matrix is represented using three vectors
        # The array of data
        self.__a = []
        # The array of row indices
        self.__i = []
        # The array of column indices
        self.__j = []

    # Reinit
    # Resize the matrix and initialize it with random values
    # n and m represent the number of rows and columns, respectively.
    # nnz_per_row is the number of non-zero elements per row
    def reinit(self, n, m, nnz_per_row):
        super().reinit(n, m)
        # The matrix has nnz_per_row entries per row; therefore, the total number of non-zero
        # entries, and the length of the a, i, and j vectors, is n * nnz_per_row
        nnz = nnz_per_row * n
        self.__a = [0] * nnz
        self.__i = [0] * nnz
        self.__j = [0] * nnz
        # Loop through rows
        for i in range(0, n):
            # Generate nnz_per_row unique column indices
            cols = random.sample(range(0, m), nnz_per_row)
            # Loop through the generated column indices
            for j in range(0, nnz_per_row):
                # The index of the current entry
                index = nnz_per_row * i + j
                # Generate random entry
                self.__a[index] = Complex(random.random(), random.random())
                # Row index
                self.__i[index] = i
                # Column index
                self.__j[index] = cols[j]

    # Set entry
    # Sets the value of the given entry
    # If the entry does not exist, it is appended at the end
    def set(self, i, j, val):
        found = False
        # The total number of non-zero entries
        nnz = len(self.__a)
        for k in range(0, nnz):
            # Check if the row index matches
            if (self.__i[k] == i):
                # Check if the column index matches
                if (self.__j[k] == j):
                    # Set value
                    self.__a[k] = val
                    # Entry found, break
                    found = True
                    break
        # If entry not found
        if (not found):
            # Entry not found, append at the end
            self.__i.append(i)
            self.__j.append(j)
            self.__a.append(val)
        
    # Entry access through the parentheses operator
    # Returns the value at the given row and column
    def __call__(self, i, j):
        # The total number of non-zero entries
        nnz = len(self.__a)
        for k in range(0, nnz):
            # Check if the row index matches
            if (self.__i[k] == i):
                # Check if the column index matches
                if (self.__j[k] == j):
                    # Return the value
                    return self.__a[k]

        # If no non-zero entry at the given row and column indices is found,
        # the entry is zero
        return Complex(0., 0.)

    # Matrix-vector multiplication
    def mult(self, vec):
        # Initialize result vector to zero
        res = [Complex(0., 0.)] * self.n_rows()
        nnz = len(self.__a)
        # Loop through non-zeros
        for k in range(0, nnz):
            res[self.__i[k]] += self.__a[k] * vec[self.__j[k]]
        return res

# List of dictionaries complex sparse matrix
class LoDCMat(CMat):
    # Constructor
    # Initializes an empty matrix
    def __init__(self):
        # Call parent's constructor 
        super().__init__()
        # Initialize data
        # List of dictionaries
        self.__rows = []

    # Reinit
    # Resize the matrix and initialize it with random values
    # n and m represent the number of rows and columns, respectively.
    # nnz_per_row is the number of non-zero elements per row
    def reinit(self, n, m, nnz_per_row):
        super().reinit(n, m)
        # Create a dictionary per row
        self.__rows = [dict() for _ in range(0, n)]
        # Loop through rows
        for i in range(0, n):
            # Generate nnz_per_row unique column indices
            cols = random.sample(range(0, m), nnz_per_row)
            # Loop through the generated column indices
            for j in range(0, nnz_per_row):
                # Generate random entry and add it to the corresponding row
                self.__rows[i][cols[j]] = Complex(random.random(), random.random())

    # Set entry
    # Sets the value of the given entry
    # If the entry does not exist, it is appended at the end
    def set(self, i, j, val):
        self.__rows[i][j] = val
        
    # Entry access through the parentheses operator
    # Returns the value at the given row and column
    def __call__(self, i, j):
        # Entry exists
        if j in self.__rows[i].keys():
            return self.__rows[i][j]
        # Entry does not exist
        else:
            return Complex(0., 0.)

    # Matrix-vector multiplication
    def mult(self, vec):
        # Initialize result vector to zero
        n_rows = self.n_rows()
        res = [Complex(0., 0.)] * n_rows
        # Loop through rows
        for i in range(0, n_rows):
            # Loop through row non-zero entries
            for j, val in self.__rows[i].items():
                res[i] += val * vec[j]
        return res

If you have completed the task up to this point, congratulations! You have created a Python package. The package is called `sp` and include two subpackages `mat` and `cmat`, ...

Before you can import and use the `sp` package, however, it has to be placed somewhere the Python interpreter can find it.

If you have installed Python through Anaconda, you can put the `sp` directory in the following location:

`/Users/<name>/anaconda3/bin/python`

## Homework

If you remember from the previous weeks, we saw that it was sometimes useful to convert matrices from one format to another. In particular, we implemented conversion functions from the coordinate format to both dense and LoD matrices for complex numbers, as seen below.

In [ ]:
# Converts a given coordinate format complex matrix to a dense complex matrix
def cooc_to_densec(coo):
    # Create dense matrix
    dense = DenseCMat()
    # Get number of rows and columns
    n_rows = coo.n_rows()
    n_cols = coo.n_cols()
    dense.reinit(n_rows, n_cols)
    # Loop through rows
    for i in range(0, n_rows):
        # Loop through columns
        for j in range(0, n_cols):
            # Set dense matrix value at position i,j
            dense.set(i, j, coo(i, j))

    # Return result
    return dense

# Converts a given coordinate format complex matrix to an LoD complex matrix
def cooc_to_lodc(coo):
    # Create dense matrix
    lod = LoDCMat()
    # Get number of rows and columns
    n_rows = coo.n_rows()
    n_cols = coo.n_cols()
    lod.reinit(n_rows, n_cols, 0)
    # Loop through rows
    for i in range(0, n_rows):
        # Loop through columns
        for j in range(0, n_cols):
            # Set dense matrix value at position i,j
            lod.set(i, j, coo(i, j))

    # Return result
    return lod

It is then natural to include these functions in our package under a `utility` module in the corresponding subpackage.

Your task is then to first implement these functions also for real numbers, which should be relatively straightforward and subsequently integrate them in the `sp` package.

The second task revolves around creating an appropriate visualization method for matrices. In particular, remember that adjacency matrices represent a graph underneath. It is, in turn, rather natural and useful to visualize graphs. You may already see where this is headed. Given that visualization is inevitably a creative task, I will leave the details up to you. Likewise, another question to be answered is (i) where this new functionality should be located in the package and perhaps more importantly (ii) if the visualization module needs to be implemented for real and complex numbers separately.

Copyright 2024 &copy; Manuel Saberi, High Performance Computing, Ruhr University Bochum. All rights reserved. No part of this notebook may be reproduced, distributed, or transmitted in any form or by any means, including photocopying, recording, or other electronic or mechanical methods, without the prior written permission of the publisher.